# Unsupervised Learning: Clustering Lab

This notebook applies unsupervised clustering algorithms — **K-means** and **Hierarchical Agglomerative Clustering (HAC)** — to the Abalone, Iris, and Wine datasets, covering:

- Baseline K-means clustering and interpreting cluster centers, inertia, and silhouette scores
- Comparing HAC linkage types (single, complete, average, ward) against K-means
- The effect of `n_init` and K-means++ initialization on cluster quality
- Choosing an optimal *k* using silhouette analysis across multiple values
- A head-to-head comparison of K-means and HAC on the same dataset

Datasets are loaded directly from public URLs so the notebook runs end-to-end without any local setup.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd
from scipy.io import arff

In [ ]:
import urllib.request, io

def load_arff_url(url):
    """Load an .arff file directly from a URL (no local download needed)."""
    text = urllib.request.urlopen(url).read().decode('utf-8')
    return arff.loadarff(io.StringIO(text))


## K-means and HAC on the Abalone Dataset

### K-means
I ran K-means (`n_clusters=3, init='random', n_init=1`) on a reduced version of the [Abalone dataset](https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/abalone.arff), treating the output class as an additional input feature rather than a label. For the resulting clustering I looked at the per-point cluster labels, the cluster centers, the number of iterations to converge, the total inertia (sum of squared error), and the average silhouette score.

In [ ]:
data, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/abalone.arff')

abalone = pd.DataFrame(data)

kmeans = KMeans(n_clusters=3, init='random', n_init=1)
kmeans.fit(abalone)

print("labels:", kmeans.labels_)
print("cluster centers:", kmeans.cluster_centers_)
print("iterations to converge:", kmeans.n_iter_)
print("total sum squared error of each point from its cluster center:", kmeans.inertia_)
print("total average silhouette score:", silhouette_score(abalone, kmeans.labels_))

# different k values

print('\n\nk = 5')
kmeans1 = KMeans(n_clusters=5, init='random', n_init=1)
kmeans1.fit(abalone)

print("labels:", kmeans1.labels_)
print("cluster centers:", kmeans1.cluster_centers_)
print("iterations to converge:", kmeans1.n_iter_)
print("total sum squared error of each point from its cluster center:", kmeans1.inertia_)
print("total average silhouette score:", silhouette_score(abalone, kmeans1.labels_))

print('\n\nk = 2')
kmeans2 = KMeans(n_clusters=2, init='random', n_init=1)
kmeans2.fit(abalone)

print("labels:", kmeans2.labels_)
print("cluster centers:", kmeans2.cluster_centers_)
print("iterations to converge:", kmeans2.n_iter_)
print("total sum squared error of each point from its cluster center:", kmeans2.inertia_)
print("total average silhouette score:", silhouette_score(abalone, kmeans2.labels_))

**Analysis**

After running it a few times, the number of iterations to converge ranged between 3-6 with the total average silhouette score staying around 0.5.
I then ran it with a higher k value (5) and a lower k value (2). The higher k value performed worse, with higher average iterations to converge and slightly lower silhouette score. The lower k value (k=2) actually performed better, with less iterations to converge and a higher average silhouette score of around 0.58.

### Hierarchical Agglomerative Clustering (HAC)
For comparison, I ran HAC with complete linkage and k=3 on the same Abalone dataset, again reporting the per-point cluster labels and the average silhouette score.

In [ ]:
hac = AgglomerativeClustering(n_clusters=3, linkage='complete')
hac.fit(abalone)

print("labels_:", hac.labels_)
print("silhouette_score:", silhouette_score(abalone, hac.labels_))

**Analysis**

hac produced a slightly better silhouette score (~0.540) compared to the K-means model k=3. This difference comes down to K-means relying on a single random initialization that tries to form spherical clusters, whereas HAC is a bottom-up approach that used complete linkage to forcefully group tightly bound clusters.

## K-means Clustering on the Iris Dataset
For this section I switched to the Iris dataset (excluding the output label from the input features).

### Initial Centroid Experiments
K-means results depend heavily on the initial centroids. I compared three approaches:
- Running K-means 5 times with `k=4` and `init='random', n_init=1` — different random centroids each time — to see how much the results vary run to run.
- Using scikit-learn's `n_init` parameter to automate multiple random runs and keep the best (lowest inertia) result.
- Using `init='k-means++'` (with `n_init=1`), which picks smarter initial centroids using a fast preliminary pass before running full K-means.

In [ ]:
data, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/iris.arff')

iris = pd.DataFrame(data)
iris_features = iris.drop(columns=['class'])

print("5 runs ")
for i in range(1, 6):
    km = KMeans(n_clusters=4, init='random', n_init=1)
    km.fit(iris_features)
    sil = silhouette_score(iris_features, km.labels_)
    print(f"Run {i} | inertia_: {km.inertia_:.4f} | silhouette_score: {sil:.4f}")

print("\n\nn init")
km_ninit = KMeans(n_clusters=4, init='random', n_init=5)
km_ninit.fit(iris_features)
sil_ninit = silhouette_score(iris_features, km_ninit.labels_)
print(f"inertia_: {km_ninit.inertia_:.4f} | silhouette_score: {sil_ninit:.4f}")

print("\n\nk means++ ")
km_pp = KMeans(n_clusters=4, init='k-means++', n_init=1)
km_pp.fit(iris_features)
sil_pp = silhouette_score(iris_features, km_pp.labels_)
print(f"inertia_: {km_pp.inertia_:.4f} | silhouette_score: {sil_pp:.4f}")

**Analysis**

Setting the n_init parameter automates multiple random runs behind the scenes and returns the grouping with the lowest sum of squared errors. In contrast, k-means++ spaces out the initial centroids from the very start which achieves an effective grouping in just a single attempt.

### Silhouette Graphs Across *k*
Using the [Yellowbrick](https://www.scikit-yb.org/en/latest/quickstart.html) `SilhouetteVisualizer`, I generated silhouette graphs for K-means clusterings with *k* = 2 through 6 (using the default `n_init=10`), printing the inertia and total silhouette score for each, then compared them to choose the best *k*.

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

for k in range(2, 7):
    km = KMeans(n_clusters=k, init='k-means++', n_init=10)

    visualizer = SilhouetteVisualizer(km, colors='yellowbrick')
    visualizer.fit(iris_features)
    visualizer.show()

    print(f"k={k}  inertia_: {km.inertia_:.4f} silhouette_score: {silhouette_score(iris_features, km.labels_):.4f}")
    print()

**Analysis**

I would choose k=3. Even though k=2 has a higher total silhouette score, k=3 has much more uniform cluster sizes and all of its clusters still safely cross the average score threshold.

## Iris Clustering with HAC
I repeated the same *k* = 2–6 sweep with HAC on the Iris dataset, this time comparing all four linkage options — single, average, complete, and ward — in a single table of silhouette scores.

In [ ]:
k_list = [2, 3, 4, 5, 6]
linkage_types = ['single', 'average', 'complete', 'ward']

results_df = pd.DataFrame(index=['k=2', 'k=3', 'k=4', 'k=5', 'k=6'])
results_df.index.name = 'k'

for link in linkage_types:
    scores = []
    for k in k_list:
        hac = AgglomerativeClustering(n_clusters=k, linkage=link)
        hac.fit(iris_features)
        sil = silhouette_score(iris_features, hac.labels_)
        scores.append(round(sil, 4))
    results_df[link] = scores

print(results_df)

**Analysis**

For all linkage types, the silhouette scores decrease as k increases from 2 to 6 which suggests that the iris data is best separated into fewer clusters. At k=2, the single, average, and ward linkage types all tie at 0.6864. At k=3, ward linkage produces a score of 0.5541 and does better than single linkage. Compared to K-means, HAC with ward linkage at k=3 produces fairly good silhouette scores, but K-means with k-means++ initialization generally performs better.

## Comparing K-means and HAC on a New Dataset
To directly compare the two algorithms on data neither had seen yet in this notebook, I pulled the [UCI Wine dataset](https://archive.ics.uci.edu/dataset/109/wine) via `ucimlrepo`, ran both algorithms a few times with different parameters, and reported a representative example of cluster labels and silhouette scores for each, along with a silhouette graph for one reasonable *k*.

In [ ]:
!pip install ucimlrepo -q
from ucimlrepo import fetch_ucirepo

In [ ]:
wine = fetch_ucirepo(id=109)
wine_features = wine.data.features

print("\nk means")
values = [
    {'k': 2, 'init': 'random', 'n_init': 1},
    {'k': 3, 'init': 'random', 'n_init': 1},
    {'k': 3, 'init': 'k-means++', 'n_init': 10}]
for val in values :
    km = KMeans(n_clusters=val['k'], init=val['init'], n_init=val['n_init'])
    km.fit(wine_features)
    sil = silhouette_score(wine_features, km.labels_)
    print(f"k={val['k']}, init={val['init']}, n_init={val['n_init']} inertia: {km.inertia_:.2f} silhouette: {sil:.4f}")
    print(f"labels: {km.labels_}\n")

print("\ngraph")
km_best = KMeans(n_clusters=3, init='k-means++', n_init=10)
viz = SilhouetteVisualizer(km_best, colors='yellowbrick')
viz.fit(wine_features)
viz.show()

print("hac")
for linkage in ['single', 'average', 'complete', 'ward']:
    hac = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    hac.fit(wine_features)
    sil = silhouette_score(wine_features, hac.labels_)
    print(f"linkage={linkage} | silhouette: {sil:.4f}")
    print(f"labels: {hac.labels_}\n")

**Analysis**

Looking for strictly the highest score, K-means at k=2 with 0.6555 and HAC with average linkage 0.6101 scored the highest. However, the true best performer is K-means with k-means++ at k=3 (0.5711) because it actually creates balanced, meaningful clusters instead of just lumping all the data into one giant group. I would choose k-means++ with k=3.